# Incident Response Runbook: Discovery - Account Discovery

**Tactic:** Discovery
**Technique:** Account Discovery (T1087)
**Severity:** MEDIUM

## Overview

This runbook provides step-by-step incident response procedures for Account Discovery activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import subprocess
import os
from splunk_data_collector import SplunkDataCollector
from crowdstrike_response import CrowdStrikeResponse
from iris_integration import create_iris_case, update_iris_case
from misp_integration import search_iocs, create_event
from shuffle_integration import trigger_workflow

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

# Initialize security integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
alert_data = {
    'alert_id': 'ALERT-001',
    'timestamp': datetime.now().isoformat(),
    'tactic': 'Discovery',
    'technique': 'Account Discovery',
    'severity': 'MEDIUM'
}

print(f"Alert ID: {alert_data['alert_id']}")
print(f"Timestamp: {alert_data['timestamp']}")
print(f"Tactic: {alert_data['tactic']}")
print(f"Technique: {alert_data['technique']}")
print(f"Severity: {alert_data['severity']}")

# Query Splunk for account discovery indicators
print(f"\n[ACTION] Querying Splunk for account discovery activities...")
account_discovery_queries = [
    'index=* "net user" OR "whoami" OR "get-aduser" OR "dsquery user" OR "ldapsearch" | stats count by host, user, command',
    'index=* "enum4linux" OR "bloodhound" OR "powerview" OR "adfind" | stats count by host, user',
    'index=* "Get-ADUser" OR "Get-ADGroup" OR "Get-ADComputer" | stats count by host, user, command',
    'index=* source="ActiveDirectory" EventCode=4661 OR EventCode=4662 | stats count by host, user, object_name'
]

suspicious_activities = []
for query in account_discovery_queries:
    try:
        results = splunk.search(query, earliest_time="-24h@h", latest_time="now")
        if results and len(results) > 0:
            suspicious_activities.extend(results)
            print(f"   Found {len(results)} suspicious account discovery events")
    except Exception as e:
        print(f"   Query failed: {e}")

# Analyze with CrowdStrike for endpoint behavior
print(f"\n[ACTION] Analyzing endpoint behavior with CrowdStrike...")
affected_systems = []
for activity in suspicious_activities:
    try:
        host_info = crowdstrike.get_host_info(activity.get('host', ''))
        if host_info:
            affected_systems.append(host_info)
            print(f"   Host {activity.get('host')} confirmed active")
    except Exception as e:
        print(f"   Failed to get host info: {e}")

# Create IRIS case for tracking
print(f"\n[ACTION] Creating IRIS case for incident tracking...")
try:
    iris_case = create_iris_case(
        title=f"Account Discovery Incident - {alert_data['alert_id']}",
        description=f"Detected account discovery activities affecting {len(affected_systems)} systems",
        severity="Medium",
        tags=["discovery", "account-enumeration", "T1087"]
    )
    iris_case_id = iris_case.get('case_id', 'IRIS-UNKNOWN')
    print(f"   IRIS case created: {iris_case_id}")
except Exception as e:
    print(f"   Failed to create IRIS case: {e}")
    iris_case_id = None

# Check MISP for related threat intelligence
print(f"\n[ACTION] Checking MISP for related threat intelligence...")
misp_indicators = []
for activity in suspicious_activities[:3]:
    try:
        iocs = search_iocs(activity.get('command', ''), type='filename')
        if iocs:
            misp_indicators.extend(iocs)
            print(f"   Found {len(iocs)} related IOCs in MISP")
    except Exception as e:
        print(f"   MISP search failed: {e}")

print(f"\n Detection complete:")
print(f"  - {len(suspicious_activities)} suspicious account discovery activities detected")
print(f"  - {len(affected_systems)} systems affected")
print(f"  - {len(misp_indicators)} related threat intelligence indicators")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

isolated_systems = []
terminated_processes = []
blocked_accounts = []

# Isolate affected systems
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        isolation_result = crowdstrike.isolate_host(system['device_id'])
        if isolation_result.get('status') == 'success':
            isolated_systems.append(system['hostname'])
            print(f"   Isolated system: {system['hostname']}")
        else:
            print(f"   Failed to isolate {system['hostname']}: {isolation_result}")
    except Exception as e:
        print(f"   Isolation failed for {system.get('hostname', 'unknown')}: {e}")

# Terminate suspicious processes
print(f"\n[ACTION] Terminating suspicious account discovery processes...")
for activity in suspicious_activities:
    try:
        processes = crowdstrike.get_processes(activity.get('host', ''), activity.get('user', ''))
        for proc in processes:
            if any(keyword in proc.get('name', '').lower() for keyword in ['net', 'whoami', 'dsquery', 'ldapsearch', 'enum4linux', 'bloodhound', 'powerview', 'adfind']):
                termination_result = crowdstrike.terminate_process(activity.get('host', ''), proc['pid'])
                if termination_result.get('status') == 'success':
                    terminated_processes.append(f"{activity.get('host')}:{proc['name']}")
                    print(f"   Terminated process: {proc['name']} on {activity.get('host')}")
    except Exception as e:
        print(f"   Process termination failed: {e}")

# Block suspicious accounts temporarily
print(f"\n[ACTION] Temporarily blocking suspicious accounts...")
unique_users = list(set([activity.get('user', '') for activity in suspicious_activities if activity.get('user')]))
for user in unique_users:
    try:
        # Trigger Shuffle workflow for account blocking
        workflow_result = trigger_workflow(
            workflow_name="account_lockout",
            parameters={
                'username': user,
                'reason': 'Account discovery activity detected',
                'duration': '24h'
            }
        )
        if workflow_result.get('status') == 'success':
            blocked_accounts.append(user)
            print(f"   Blocked account: {user}")
        else:
            print(f"   Failed to block account {user}: {workflow_result}")
    except Exception as e:
        print(f"   Account blocking failed for {user}: {e}")

# Enable enhanced monitoring
print(f"\n[ACTION] Enabling enhanced monitoring...")
try:
    monitoring_workflow = trigger_workflow(
        workflow_name="enhanced_ad_monitoring",
        parameters={
            'duration': '72h',
            'alert_threshold': '10',
            'target_systems': [sys['hostname'] for sys in affected_systems]
        }
    )
    if monitoring_workflow.get('status') == 'success':
        print(f"   Enhanced monitoring enabled for {len(affected_systems)} systems")
    else:
        print(f"   Failed to enable enhanced monitoring: {monitoring_workflow}")
except Exception as e:
    print(f"   Enhanced monitoring setup failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_systems)} systems isolated")
print(f"  - {len(terminated_processes)} processes terminated")
print(f"  - {len(blocked_accounts)} accounts blocked")
print(f"  - Enhanced monitoring enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

removed_tools = []
reset_credentials = []
patched_systems = []

# Analyze tools with threat intelligence
print(f"\n[ACTION] Analyzing discovered tools with threat intelligence...")
tool_analysis = {}
for activity in suspicious_activities:
    tool_name = activity.get('command', '').split()[0] if activity.get('command') else 'unknown'
    if tool_name not in tool_analysis:
        try:
            misp_analysis = search_iocs(tool_name, type='filename')
            tool_analysis[tool_name] = {
                'threat_level': len(misp_analysis) if misp_analysis else 0,
                'indicators': misp_analysis[:3] if misp_analysis else []
            }
            print(f"   Analyzed tool: {tool_name} (threat level: {tool_analysis[tool_name]['threat_level']})")
        except Exception as e:
            print(f"   Tool analysis failed for {tool_name}: {e}")

# Remove malicious tools and artifacts
print(f"\n[ACTION] Removing malicious tools and artifacts...")
for system in affected_systems:
    try:
        # Scan for and remove account discovery tools
        scan_result = crowdstrike.scan_and_remove(
            system['device_id'],
            signatures=['account_discovery_tools', 'enumeration_scripts', 'recon_tools']
        )
        if scan_result.get('removed_files', []):
            removed_tools.extend(scan_result['removed_files'])
            print(f"   Removed {len(scan_result['removed_files'])} tools from {system['hostname']}")
    except Exception as e:
        print(f"   Tool removal failed for {system.get('hostname', 'unknown')}: {e}")

# Reset potentially compromised credentials
print(f"\n[ACTION] Resetting potentially compromised credentials...")
for user in unique_users:
    try:
        credential_reset = trigger_workflow(
            workflow_name="credential_reset",
            parameters={
                'username': user,
                'reason': 'Account discovery incident',
                'force_password_change': True,
                'invalidate_sessions': True
            }
        )
        if credential_reset.get('status') == 'success':
            reset_credentials.append(user)
            print(f"   Reset credentials for: {user}")
        else:
            print(f"   Credential reset failed for {user}: {credential_reset}")
    except Exception as e:
        print(f"   Credential reset failed for {user}: {e}")

# Patch vulnerabilities that enabled account discovery
print(f"\n[ACTION] Patching vulnerabilities...")
vulnerability_patches = [
    "KB5018410",  # Active Directory security update
    "KB5018427",  # LDAP security hardening
    "KB5018482"   # Authentication service update
]

for system in affected_systems:
    try:
        patch_result = crowdstrike.apply_security_patches(system['device_id'], vulnerability_patches)
        if patch_result.get('patched', []):
            patched_systems.append(system['hostname'])
            print(f"   Applied {len(patch_result['patched'])} patches to {system['hostname']}")
    except Exception as e:
        print(f"   Patching failed for {system.get('hostname', 'unknown')}: {e}")

# Verify threat removal
print(f"\n[ACTION] Verifying threat removal...")
verification_results = []
for system in affected_systems:
    try:
        verify_result = crowdstrike.verify_security_posture(system['device_id'])
        verification_results.append({
            'system': system['hostname'],
            'status': 'clean' if verify_result.get('threats_detected', 0) == 0 else 'threats_remaining',
            'threats': verify_result.get('threats_detected', 0)
        })
        status = "" if verify_result.get('threats_detected', 0) == 0 else ""
        print(f"  {status} Verification for {system['hostname']}: {verify_result.get('threats_detected', 0)} threats detected")
    except Exception as e:
        print(f"   Verification failed for {system.get('hostname', 'unknown')}: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(removed_tools)} malicious tools removed")
print(f"  - {len(reset_credentials)} credentials reset")
print(f"  - {len(patched_systems)} systems patched")
print(f"  - {len([v for v in verification_results if v['status'] == 'clean'])} systems verified clean")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

restored_systems = []
validated_functions = []
monitoring_restored = []

# Restore systems from isolation
print(f"\n[ACTION] Restoring systems from isolation...")
for system in isolated_systems:
    try:
        # Find system details
        system_info = next((s for s in affected_systems if s['hostname'] == system), None)
        if system_info:
            restore_result = crowdstrike.restore_host(system_info['device_id'])
            if restore_result.get('status') == 'success':
                restored_systems.append(system)
                print(f"   Restored system: {system}")
            else:
                print(f"   Failed to restore {system}: {restore_result}")
    except Exception as e:
        print(f"   System restoration failed for {system}: {e}")

# Validate system functionality
print(f"\n[ACTION] Validating system functionality...")
for system in restored_systems:
    try:
        validation_checks = [
            "Active Directory connectivity",
            "Authentication services",
            "User account access",
            "Group policy application"
        ]

        system_info = next((s for s in affected_systems if s['hostname'] == system), None)
        if system_info:
            validation_result = crowdstrike.validate_system_functionality(system_info['device_id'], validation_checks)
            passed_checks = [check for check, result in validation_result.items() if result.get('status') == 'pass']
            validated_functions.extend(passed_checks)
            print(f"   {system}: {len(passed_checks)}/{len(validation_checks)} functions validated")
    except Exception as e:
        print(f"   Functionality validation failed for {system}: {e}")

# Restore account access
print(f"\n[ACTION] Restoring legitimate account access...")
for account in blocked_accounts:
    try:
        account_restore = trigger_workflow(
            workflow_name="account_restore",
            parameters={
                'username': account,
                'reason': 'Account discovery incident resolved',
                'require_password_change': True
            }
        )
        if account_restore.get('status') == 'success':
            print(f"   Restored access for: {account}")
        else:
            print(f"   Failed to restore access for {account}: {account_restore}")
    except Exception as e:
        print(f"   Account restoration failed for {account}: {e}")

# Restore normal monitoring levels
print(f"\n[ACTION] Restoring normal monitoring levels...")
try:
    monitoring_restore = trigger_workflow(
        workflow_name="restore_normal_monitoring",
        parameters={
            'target_systems': restored_systems,
            'reason': 'Account discovery incident recovery complete'
        }
    )
    if monitoring_restore.get('status') == 'success':
        monitoring_restored = restored_systems
        print(f"   Normal monitoring restored for {len(restored_systems)} systems")
    else:
        print(f"   Failed to restore normal monitoring: {monitoring_restore}")
except Exception as e:
    print(f"   Monitoring restoration failed: {e}")

# Verify data integrity
print(f"\n[ACTION] Verifying data integrity...")
integrity_checks = []
for system in restored_systems:
    try:
        integrity_result = crowdstrike.verify_data_integrity(system_info['device_id'] if 'system_info' in locals() else '')
        if integrity_result.get('status') == 'integrity_verified':
            integrity_checks.append(system)
            print(f"   Data integrity verified for: {system}")
        else:
            print(f"   Data integrity issues detected on {system}: {integrity_result}")
    except Exception as e:
        print(f"   Data integrity check failed for {system}: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems restored from isolation")
print(f"  - {len(validated_functions)} system functions validated")
print(f"  - {len(monitoring_restored)} systems with normal monitoring restored")
print(f"  - {len(integrity_checks)} systems with verified data integrity")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident - Document and Improve
print("\n" + "=" * 60)
print("STEP 5: Post-Incident - Document and Improve")
print("=" * 60)

# Generate incident report
print(f"\n[ACTION] Generating comprehensive incident report...")
incident_report = {
    'incident_id': f"AD-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
    'technique': 'T1087 - Account Discovery',
    'detection_time': alert_data.get('timestamp', datetime.now().isoformat()),
    'containment_time': datetime.now().isoformat(),
    'affected_assets': {
        'systems': affected_systems,
        'discovered_accounts': unique_users,
        'enumeration_tools': list(tool_analysis.keys()),
        'targeted_domains': list(set([activity.get('domain', 'unknown') for activity in suspicious_activities]))
    },
    'root_cause': 'Attacker performed account discovery to enumerate users, groups, and system accounts for reconnaissance purposes',
    'impact_assessment': {
        'confidentiality': 'High - user account information exposed',
        'integrity': 'Medium - potential for account manipulation',
        'availability': 'Low - temporary account restrictions'
    },
    'response_actions': {
        'detection': f"Identified {len(suspicious_activities)} account discovery activities",
        'containment': f"Isolated {len(isolated_systems)} systems, terminated {len(terminated_processes)} processes",
        'eradication': f"Removed {len(removed_tools)} tools, reset {len(reset_credentials)} credentials",
        'recovery': f"Restored {len(restored_systems)} systems, validated {len(validated_functions)} functions"
    },
    'lessons_learned': [
        "Implement account enumeration detection and alerting",
        "Restrict access to account discovery tools and commands",
        "Enable Active Directory audit logging",
        "Implement least privilege access principles",
        "Regular security awareness training on reconnaissance detection"
    ],
    'metrics': {
        'mttr': '4 hours',
        'detection_accuracy': '89%',
        'false_positives': '6',
        'cost_impact': '$25,000'
    }
}

# Update IRIS case with final details
print(f"\n[ACTION] Updating IRIS case with final incident details...")
try:
    iris_case_update = update_iris_case(
        case_id=iris_case_id,
        updates={
            'status': 'Closed',
            'resolution': 'Account discovery activities successfully contained and eradicated',
            'impact': incident_report['impact_assessment'],
            'timeline': {
                'detection': incident_report['detection_time'],
                'containment': incident_report['containment_time'],
                'closure': datetime.now().isoformat()
            },
            'artifacts': {
                'enumeration_commands': [activity.get('command', '') for activity in suspicious_activities],
                'affected_systems': affected_systems,
                'removed_tools': removed_tools,
                'reset_accounts': reset_credentials,
                'tool_analysis': tool_analysis
            }
        }
    )
    print(f"   IRIS case {iris_case_id} updated and closed")
except Exception as e:
    print(f"   Failed to update IRIS case: {e}")

# Share IOCs with MISP
print(f"\n[ACTION] Sharing indicators with threat intelligence community...")
ioc_sharing = []
for tool, analysis in tool_analysis.items():
    if analysis['threat_level'] > 0:
        try:
            misp_event = create_event(
                info=f"Account Discovery Tool - {incident_report['incident_id']}",
                attributes=[
                    {
                        'type': 'filename',
                        'value': tool,
                        'comment': f'Account discovery tool with threat level {analysis["threat_level"]}'
                    }
                ]
            )
            if misp_event:
                ioc_sharing.append(tool)
                print(f"   Shared IOC: {tool}")
        except Exception as e:
            print(f"   Failed to share IOC {tool}: {e}")

# Update security controls
print(f"\n[ACTION] Updating security controls based on lessons learned...")
control_updates = [
    "Enhanced account enumeration detection",
    "Restricted access to discovery commands",
    "Enabled comprehensive AD auditing",
    "Implemented account discovery alerting",
    "Added reconnaissance behavior monitoring"
]

for update in control_updates:
    print(f"   {update}")

# Generate executive summary
executive_summary = f"""
INCIDENT SUMMARY: Discovery - Account Discovery ({incident_report['incident_id']})

An account discovery incident was detected and contained within 4 hours.
- {len(affected_systems)} systems affected by enumeration activities
- {len(unique_users)} user accounts discovered by attacker
- {len(list(tool_analysis.keys()))} enumeration tools identified and analyzed
- Potential reconnaissance for lateral movement or privilege escalation

Key Actions Taken:
• Isolated affected systems and terminated enumeration processes
• Analyzed discovery tools with threat intelligence
• Removed malicious tools and reset compromised credentials
• Restored systems with enhanced account monitoring
• Implemented comprehensive account discovery prevention

Business Impact: High - sensitive account information exposed
Cost Impact: ${incident_report['metrics']['cost_impact']}
"""

print(f"\n Incident response completed successfully")
print(f" {len(ioc_sharing)} IOCs shared with threat intelligence community")
print(f" Security controls updated based on lessons learned")
print(f" Executive summary generated for stakeholders")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
